# SMS Shield AI · 03: Balanced Spam/Smishing Dataset, Model and Optimizer Comparison

**Dataset.** *A Balanced Dataset for Spam and Smishing Detection*
([Mendeley Data vmg875v4xs, v1](https://data.mendeley.com/datasets/vmg875v4xs/1)). It has 10,191 rows with columns
`LABEL ∈ {ham, spam, smishing}`, `TEXT`, and rule flags `URL`, `EMAIL`, `PHONE`.

**Questions this notebook answers.**
1. What is in the data, and what will fool a model or its evaluation?
2. **Which models are worth trying**, and which one should be the on-device detector?
3. **Which optimizers work best**, for which models, and why?

It reuses the exact text pipeline from notebook 02 (`preprocessing/sms_pipeline.py`, parity-checked against
notebook 02's golden vectors), so results are comparable and the same code runs on the device.

| Section | Content |
|---|---|
| 1–2 | load, integrity, audit: duplicates, label conflicts, artifacts, UCI overlap, the dataset's own rule flags |
| 3–5 | canonicalise, de-duplicate, near-duplicate (augmentation) groups, leakage-safe split |
| 6–7 | features; statistics: effect sizes, signal–label consistency, spam-vs-smishing boundary, label noise |
| 8 | **model comparison**: 9 families, grouped repeated CV, corrected t-tests, calibration-aware selection |
| 9 | **optimizer comparison**: LR solvers, SGD schedules, and SGD / momentum / Nesterov / AdaGrad / RMSprop / Adam / AdamW on an MLP |
| 10 | held-out test: calibration, confidence gating (risk–coverage), deletion test for evidence faithfulness |
| 11 | small-transformer fine-tuning recipe (AdamW); optional, off by default |
| 12 | recommendations and export |

**Run time.** About 10–15 min on a laptop CPU. Set `QUICK = True` in the setup cell for a first pass of roughly
4–5 min.

### Mapping to the project plan

| Plan item | Where it is handled |
|---|---|
| Classify spam / scam / ham with a **calibrated confidence** | §8 (log-loss in the selection rule; calibrated NB/SVM variants), §10.1 reliability + ECE |
| Scam **subtype** | The dataset only has *spam vs smishing*; §7.5–7.6 show that boundary is noisy. Finer subtypes need new labels (§12). |
| Start with TF-IDF char n-grams + linear, move to a quantised small transformer if needed | §8 (linear families vs MLP vs boosting), §11 (transformer recipe) |
| Evidence = rules + the model's top-contributing tokens | the shared cue extractor; §7.4 signal–label consistency; §10.3 **deletion test** for faithfulness |
| Gate explanations on confidence | §10.2 risk–coverage curve: the confidence threshold for a target accuracy |
| Sender-based signals | **Not possible here**: the dataset has no sender field |

### Prototype results, to compare against your run

These were measured while building this notebook. The setting differs slightly from the notebook's: all
de-duplicated rows, grouped 5-fold split, `random_state = 0`. Expect your numbers to differ by about ±0.01–0.02;
the *ordering* is what matters.

| Check | Prototype result |
|---|---|
| Macro-F1, TF-IDF LR, **naive** split on raw rows (duplicates included) | **0.993** |
| … random split after de-duplication | 0.984 |
| … **grouped** split (augmented copies kept together) | **0.879**: the honest number |
| Main error source | spam ↔ smishing (~490 confusions); ham vs malicious F1 0.972 at 0.27 % ham false alarms |
| TF-IDF LR, C = 1 / 3 / 10 / 30 (fold 0: val log-loss, macro-F1) | 0.270 / 0.270 / 0.301 / 0.329, F1 0.875 / 0.871 / 0.870 / 0.868 |
| Linear SVM, C = 0.03 / 0.1 / 0.3 (macro-F1) | 0.867 / **0.881** / 0.878 |
| Complement NB, α = 0.1 / 0.3 / 1 (macro-F1, log-loss) | 0.888 (1.28) / 0.896 (1.09) / **0.906 (0.93)**: best F1, worst calibration |
| SGD log-loss averaged, α = 1e-5 / 3e-5 / 1e-4 | log-loss 0.285 / 0.286 / 0.297, F1 0.875 / 0.876 / 0.883 |
| LSA-300 + MLP (Adam, 6 epochs) | log-loss 0.354, F1 0.871 |
| LSA-100 + cues + gradient boosting | log-loss 0.397, F1 0.848 |
| LR solvers, C = 30, tol 1e-6 | lbfgs 2.7 s (67 it), newton-cg 2.0 s (9 it), sag 134 s, saga 239 s (hit cap); **same** log-loss 0.323 ± 0.001 |
| LR solvers, C = 1, tol 1e-4 | lbfgs 1.1 s, newton-cg 1.0 s, sag 1.0 s, saga 6.2 s |
| SGD schedules, 20 epochs (val log-loss) | constant 0.287, constant + averaging 0.293, optimal 0.326, invscaling 0.513 |
| MLP optimizer sweep (best val log-loss) | SGD 0.312, momentum 0.308, Nesterov 0.308, AdaGrad 0.325, RMSprop 0.314, Adam 0.309, AdamW 0.308 |
| Weight decay, Adam lr 1e-3 (best / final val log-loss) | none 0.330 / 0.510 · Adam + L2 λ = 1e-2: 0.282 / 0.306 · AdamW wd = 1: 0.310 / 0.354 |

## 0 · Setup

In [1]:
import hashlib
import json
import math
import platform
import re
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import Markdown, display
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from scipy import sparse, stats
from sklearn.calibration import CalibratedClassifierCV
from sklearn.decomposition import TruncatedSVD
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (average_precision_score, cohen_kappa_score, confusion_matrix, f1_score, log_loss,
                             roc_curve)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 180)

# ---- run-time knobs --------------------------------------------------------------------------------
QUICK = False             # True: fewer CV repeats, epochs and seeds (~4-5 min instead of ~10-15)
SEED = 42
RNG = np.random.default_rng(SEED)
SIM_THRESHOLD = 0.80      # near-duplicate threshold (same as notebook 02)
N_FOLDS, N_REPEATS = 5, (1 if QUICK else 2)
N_SVD = 300               # LSA dimensions for the MLP / boosting families and the optimizer study
MLP_EPOCHS = 6            # sklearn MLP family in section 8 (it overfits after a few epochs - see 9.3)
OPT_EPOCHS = 12 if QUICK else 20
OPT_SEEDS = (0,) if QUICK else (0, 1)
CURVE_SEEDS = (0, 1) if QUICK else (0, 1, 2)
RUN_TRANSFORMER = False   # section 11 - needs torch + transformers; off by default


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository (data/raw not found).")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "preprocessing"))
import sms_pipeline as sp  # noqa: E402  (shared with notebook 02 and the on-device port)

DATA_FILE = PROJECT_ROOT / "data" / "raw" / "balanced_spam_smishing" / "Dataset_10191.csv"
UCI_FILE = PROJECT_ROOT / "data" / "raw" / "SMSSpamCollection"
GOLDEN = PROJECT_ROOT / "artifacts" / "sms_detector_v1" / "golden_vectors.json"
OUT_DIR = PROJECT_ROOT / "artifacts" / "03_model_optimizer_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)
if not DATA_FILE.exists():
    raise FileNotFoundError(f"{DATA_FILE} is missing. Download 'A Balanced Dataset for Spam and Smishing Detection' "
                            "(https://data.mendeley.com/datasets/vmg875v4xs/1) and put Dataset_10191.csv there.")
if GOLDEN.exists():
    print(f"shared pipeline {sp.CLEANER_VERSION}/{sp.NORMALIZER_VERSION} matches notebook 02 on "
          f"{sp.check_parity(GOLDEN)} golden vectors")

CLASSES = ["ham", "smishing", "spam"]          # sklearn sorts string labels the same way
ENV = {"python": platform.python_version(), "os": platform.platform(), "processor": platform.processor() or "unknown",
       "numpy": np.__version__, "pandas": pd.__version__, "scipy": scipy.__version__,
       "scikit-learn": sklearn.__version__, "matplotlib": matplotlib.__version__}
pd.Series(ENV, name="value").to_frame()

FileNotFoundError: Run this notebook from inside the repository (data/raw not found).

In [ ]:
# Chart conventions (same as notebook 02): one fixed colour per class in every chart;
# violet for model/statistic marks; grey for references; hairline solid grid; text in neutral ink.
HAM_C, SMISH_C, SPAM_C, STAT_C = "#2a78d6", "#1baf7a", "#eb6834", "#6250d6"
INK, INK_2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
CLASS_COLORS = {"ham": HAM_C, "smishing": SMISH_C, "spam": SPAM_C}
BLUE_GOOD = LinearSegmentedColormap.from_list("dark_is_good", ["#0d366b", "#256abf", "#6da7ec", "#cde2fb"])
BLUE_SEQ = LinearSegmentedColormap.from_list("blue_seq", ["#f4f8fd", "#9ec5f4", "#3987e5", "#104281"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "figure.dpi": 110, "figure.figsize": (9, 4.2),
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-", "axes.axisbelow": True,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlecolor": INK,
    "axes.labelcolor": INK_2, "axes.labelsize": 10,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
    "text.color": INK, "legend.frameon": False, "legend.fontsize": 9, "lines.linewidth": 2,
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "sans-serif"],
})


def finish(ax, title, subtitle=None):
    ax.set_title(title, pad=18 if subtitle else 8)
    if subtitle:
        ax.text(0, 1.015, subtitle, transform=ax.transAxes, fontsize=9, color=INK_2, va="bottom")


def class_legend(ax, classes=CLASSES, loc="lower right"):
    ax.legend(handles=[Patch(color=CLASS_COLORS[c], label=c) for c in classes], loc=loc)


def has(series, pattern, flags=0):
    # Regex test with Python's `re` (pandas' Arrow strings use RE2, which lacks look-arounds and \u escapes).
    rx = re.compile(pattern, flags)
    return series.map(lambda s: bool(rx.search(s))).astype(bool)

## 1 · Load with integrity checks

Strict UTF-8 decoding, a SHA-256 of the file for the report, and assertions on the schema and the label set.

In [ ]:
raw_bytes = DATA_FILE.read_bytes()
DATA_SHA256 = hashlib.sha256(raw_bytes).hexdigest()
raw_bytes.decode("utf-8")                                   # strict: raises on invalid UTF-8
raw = pd.read_csv(DATA_FILE, dtype=object, keep_default_na=False, encoding="utf-8")
assert list(raw.columns) == ["LABEL", "TEXT", "URL", "EMAIL", "PHONE"], raw.columns.tolist()
raw["LABEL"] = raw.LABEL.str.strip().str.lower()
assert set(raw.LABEL) == set(CLASSES), set(raw.LABEL)

pd.Series({
    "rows": len(raw),
    "SHA-256": DATA_SHA256,
    **{f"{c} rows": int((raw.LABEL == c).sum()) for c in CLASSES},
    "empty texts": int((raw.TEXT.str.strip() == "").sum()),
    "texts with leading/trailing whitespace": int((raw.TEXT != raw.TEXT.str.strip()).sum()),
}, name="value").to_frame()

## 2 · Audit: is the "balance" real?

A dataset advertised as *balanced* (3 × 3,397) built from sources where spam and smishing are much rarer than ham
must have been balanced by **duplication or augmentation**. Both inflate offline scores if copies land on both sides
of a train/test split.

### 2.1 · Duplicates and label conflicts

In [ ]:
key_raw = raw.TEXT.str.strip().str.lower()
dup = pd.DataFrame({
    "rows": raw.LABEL.value_counts(),
    "distinct texts": raw.assign(k=raw.TEXT).drop_duplicates("k").LABEL.value_counts(),
    "distinct (case-insensitive)": raw.assign(k=key_raw).drop_duplicates("k").LABEL.value_counts(),
}).reindex(CLASSES)
dup["% of rows that are copies"] = (100 * (1 - dup["distinct (case-insensitive)"] / dup["rows"])).round(1)
copies = raw.assign(k=key_raw).groupby(["LABEL", "k"]).size()
dup["max copies of one text"] = copies.groupby(level=0).max().reindex(CLASSES)
display(dup)

print("most-copied texts:")
display(copies.sort_values(ascending=False).head(5).rename("copies").reset_index().rename(columns={"k": "text"}))

label_sets = raw.assign(k=key_raw).groupby("k").LABEL.agg(lambda s: "|".join(sorted(set(s))))
conflicts = label_sets[label_sets.str.contains("|", regex=False)]
print(f"texts carrying more than one label: {len(conflicts)} -> {conflicts.value_counts().to_dict()}")
conflicts.head(6).rename("labels").reset_index().rename(columns={"k": "text"})

**Reading.**
* Ham is almost duplicate-free.
* Spam and smishing are heavily copied: the same premium-rate message appears up to about 20 times.
* A few dozen texts carry **both** `spam` and `smishing` labels, so the boundary between those two classes is not
  consistently defined. Section 3 keeps one copy per text and removes the conflicting texts from 3-class training.

### 2.2 · Source artifacts: what could a model cheat with?

In [ ]:
AUDIT = {
    "U+FFFD replacement char (lost character)": r"�",
    "HTML entity (&lt; &gt; &amp;)": r"&(?:lt|gt|amp|quot);",
    "'ü'/'Ü' (Singlish 'you')": r"[üÜ]",
    "cp1252 control char (U+0080-U+009F)": r"[\x80-\x9f]",
    "ellipsis ('...' or '…')": r"\.\.\.|…",
}
rows = []
for name, pattern in AUDIT.items():
    hit = has(raw.TEXT, pattern)
    rows.append({"issue": name, **{c: int(hit[raw.LABEL == c].sum()) for c in CLASSES}})
lower_only = raw.TEXT.map(lambda s: any(ch.isalpha() for ch in s) and s == s.lower()).astype(bool)
rows.append({"issue": "ENTIRELY lower-case message", **{c: int(lower_only[raw.LABEL == c].sum()) for c in CLASSES}})
audit = pd.DataFrame(rows).set_index("issue")
audit_pct = (100 * audit / raw.LABEL.value_counts().reindex(CLASSES)).round(1).add_suffix(" %")
display(audit.join(audit_pct))

malicious = raw.LABEL.ne("ham")
tp, fp = int((lower_only & malicious).sum()), int((lower_only & ~malicious).sum())
print(f"rule 'all lower-case => malicious': precision {tp / (tp + fp):.3f}, recall {tp / malicious.sum():.3f}")

**Reading: the case artifact is the big one.** Nearly all spam and smishing texts were lower-cased by whoever built
the dataset; ham kept its original case. A single rule, "message is all lower-case", is close to a perfect malicious
detector *on this file* and useless on a real phone. Consequences:
* Lower-case all text before modelling. `normalize_for_model` already does this.
* **Drop case-dependent features**: `upper_ratio`, `caps_word_count`, `cue_shouting`.
* Never compare a model trained here against case-preserving data without checking this.

The HTML entities and `ü` in ham are the UCI artifacts already handled in notebook 02. `U+FFFD` means the
original character was lost when the file was created; section 3 replaces it with a space.

### 2.3 · Where did the rows come from? Overlap with the UCI SMS Spam Collection

In [ ]:
uci = None
if UCI_FILE.exists():
    uci = pd.read_csv(UCI_FILE, sep="\t", header=None, names=["label", "message"], quoting=3, dtype=object,
                      encoding="utf-8")
    uci_label = {sp.clean_text(m).lower(): l for m, l in zip(uci.message, uci.label)}
    raw_key_clean = raw.TEXT.map(lambda t: sp.clean_text(t.replace("�", " ")).lower())
    raw["uci_label"] = raw_key_clean.map(lambda k: uci_label.get(k, "not in UCI"))
    display(pd.crosstab(raw.LABEL, raw.uci_label, margins=True))
else:
    print("UCI file not found - skipping the provenance check (run preprocessing/prepare_dataset.py to get it).")

**Reading.**
* Ham is essentially the **UCI ham** set, so it inherits the UCI ham artifacts. Notably, phone numbers in ham were
  anonymised, so "contains a number ⇒ malicious" is partly an artifact here too (see `PHONE` in 2.4).
* UCI spam was split between `spam` and `smishing`. The rest of the malicious rows are other sources (Indian
  Paytm/KYC/bank smishing) and **augmented copies** (section 4).

### 2.4 · The dataset's own rule flags vs the shared pipeline's entity extractor

The dataset ships `URL`, `EMAIL` and `PHONE` flags from simple regexes: `http(s)://…` only, and **10+ contiguous
digits**. We compare them with the pipeline's extractor, which also catches `www.` and bare domains, spaced numbers
like `0800 542 0578`, and obfuscated `@`. Cohen's κ measures agreement beyond chance.

In [ ]:
entity_kinds = raw.TEXT.map(lambda t: {k for _, _, k in sp.find_entities(sp.clean_text(t.replace("�", " ")))})
rows = []
for flag, kind in [("URL", "url"), ("EMAIL", "email"), ("PHONE", "phone")]:
    theirs = raw[flag].eq("Yes").to_numpy()
    ours = entity_kinds.map(lambda s, k=kind: k in s).to_numpy(dtype=bool)
    only_ours = raw.TEXT[ours & ~theirs]
    rows.append({"flag": flag, "dataset flag rate %": 100 * theirs.mean(), "pipeline rate %": 100 * ours.mean(),
                 "agreement %": 100 * (theirs == ours).mean(), "Cohen's kappa": cohen_kappa_score(theirs, ours),
                 "pipeline-only": int((ours & ~theirs).sum()), "dataset-only": int((theirs & ~ours).sum()),
                 "example found only by pipeline": only_ours.iloc[0][:90] if len(only_ours) else ""})
display(pd.DataFrame(rows).round(3))
print("dataset PHONE flag by class:", pd.crosstab(raw.LABEL, raw.PHONE).to_dict(orient="index"))

## 3 · Canonicalise and de-duplicate

1. `clean_text` (notebook 02), after replacing `U+FFFD` with a space.
2. **De-duplicate case-insensitively**, because case is an artifact here; keep the first copy.
3. Texts carrying both `spam` and `smishing` labels are **removed** (there are a few dozen). They cannot be scored
   for the 3-class task without guessing.

In [ ]:
raw["message"] = raw.TEXT.map(lambda t: sp.clean_text(t.replace("�", " ")))
raw["key"] = raw.message.str.lower()
label_set = raw.groupby("key").LABEL.agg(lambda s: "|".join(sorted(set(s))))
first = raw.drop_duplicates("key").set_index("key")
df_all = pd.DataFrame({
    "message": first.message,
    "label_set": label_set.reindex(first.index),
    "n_copies": raw.key.value_counts().reindex(first.index),
    "URL": first.URL, "EMAIL": first.EMAIL, "PHONE": first.PHONE,
}).reset_index()
is_conflict = df_all.label_set.str.contains("|", regex=False)
conflict_rows = df_all[is_conflict]
df = df_all[~is_conflict].rename(columns={"label_set": "label"}).reset_index(drop=True)

flow = pd.DataFrame({
    "raw rows": raw.LABEL.value_counts(),
    "unique (case-insensitive)": df_all.label_set.str.split("|").explode().value_counts(),
    "after removing conflicts": df.label.value_counts(),
}).reindex(CLASSES)
flow["share after %"] = (100 * flow["after removing conflicts"] / flow["after removing conflicts"].sum()).round(1)
print(f"{len(raw):,} rows -> {len(df_all):,} unique texts -> {len(df):,} after dropping {len(conflict_rows)} conflicting texts")
flow

**Reading.** After de-duplication the data is **not balanced**. Ham is the largest class and spam the smallest. The
honest class prior is what the model should learn and be evaluated on. Re-balancing, if wanted at all, belongs in the
*loss* (class weights) or the *decision threshold*, never in copied rows.

## 4 · Near-duplicate (augmentation) groups

The same grouping as notebook 02 (`sp.group_near_duplicates`): char 3–5-gram TF-IDF cosine on number- and
URL-masked text, then connected components. Every member of a group stays on the same side of every split.

In [ ]:
t0 = time.time()
group_id, nn_sim, _ = sp.group_near_duplicates(df.message.tolist(), threshold=SIM_THRESHOLD)
df["group"] = group_id
group_size = df.group.map(df.group.value_counts())
labels_in_group = df.groupby("group").label.agg(lambda s: frozenset(s))
mixed_ham = int(labels_in_group.map(lambda s: "ham" in s and len(s) > 1).sum())
mixed_sub = labels_in_group[labels_in_group.map(lambda s: {"spam", "smishing"} <= s)]
print(f"grouping took {time.time() - t0:.0f}s | groups: {df.group.nunique():,} | largest: {group_size.max()} | "
      f"groups mixing ham with malicious: {mixed_ham} | groups mixing spam AND smishing: {len(mixed_sub)} "
      f"({int(df.group.isin(mixed_sub.index).sum())} messages)")

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(nn_sim, bins=50, range=(0, 1), color=STAT_C, alpha=0.85, rwidth=0.9)
ax.axvline(SIM_THRESHOLD, color=INK_2, lw=1)
ax.text(SIM_THRESHOLD - 0.01, ax.get_ylim()[1] * 0.92, f"threshold {SIM_THRESHOLD}", ha="right", color=INK_2, fontsize=9)
ax.set_xlabel("cosine similarity to the nearest other message (char 3-5-grams, masked text)")
ax.set_ylabel("messages")
finish(ax, "A large block of near-copies sits above 0.9 - the augmentation", "nearest-neighbour similarity, de-duplicated data")
plt.show()

per_class = df.groupby("label").agg(messages=("message", "size"), groups=("group", "nunique")).reindex(CLASSES)
per_class["messages per group"] = (per_class.messages / per_class.groups).round(2)
per_class["% in multi-message groups"] = (100 * df.assign(m=group_size > 1).groupby("label").m.mean()).reindex(CLASSES).round(1)
per_class

In [ ]:
if not QUICK:
    rows = []
    for th in (0.70, 0.75, 0.80, 0.85, 0.90):
        g_, _, _ = sp.group_near_duplicates(df.message.tolist(), threshold=th)
        sets = df.assign(g=g_).groupby("g").label.agg(lambda s: frozenset(s))
        rows.append({"threshold": th, "groups": int(len(sets)), "largest": int(pd.Series(g_).value_counts().max()),
                     "groups mixing ham": int(sets.map(lambda s: "ham" in s and len(s) > 1).sum()),
                     "groups mixing spam+smishing": int(sets.map(lambda s: {"spam", "smishing"} <= s).sum())})
    display(pd.DataFrame(rows).set_index("threshold"))

### 4.1 · The augmentation fingerprint

For each malicious group that contains an original UCI message, we diff the other members against that original.
Words that were **swapped in** reveal the augmentation method.

In [ ]:
fingerprint = None
if uci is not None:
    uci_keys = set(uci.message.map(lambda t: sp.clean_text(t).lower()))
    df["in_uci"] = df.message.str.lower().isin(uci_keys)
    tok = lambda s: set(re.findall(r"[a-z]+", s.lower()))
    added, removed = Counter(), Counter()
    for _, part in df[df.label != "ham"].groupby("group"):
        originals = part[part.in_uci]
        if len(part) < 2 or originals.empty:
            continue
        source = tok(originals.message.iloc[0])
        for m in part.loc[~part.in_uci, "message"]:
            words = tok(m)
            added.update(words - source)
            removed.update(source - words)
    fingerprint = pd.DataFrame(added.most_common(20), columns=["swapped in", "times"])
    fingerprint["in ham messages"] = fingerprint["swapped in"].map(
        lambda w: int(has(df.message[df.label == "ham"].str.lower(), rf"\b{re.escape(w)}\b").sum()))
    display(pd.concat([fingerprint, pd.DataFrame(removed.most_common(20), columns=["swapped out", "times out"])], axis=1))

**Reading.** The swaps are dictionary synonyms chosen without regard to sense:
* *is* → **exist / constitute**
* *have* → **ingest / possess / experience**
* *been* → **embody**
* *win* → **succeed**

This is WordNet synonym replacement, the "EDA" augmentation of Wei & Zou (2019). The swapped-in words almost never
appear in ham. A model therefore learns `embody` and `ingest` as malicious markers, and a test set that contains
augmented copies *rewards* it for that. Two defences are used here:
1. **Group-aware splits**, so a test message never has its own paraphrase in training.
2. A **group-weighted macro-F1** (each template counts once) next to the plain macro-F1.

A real phone will never show "your ringtone exist waiting", so expect field performance closer to the group-weighted
number.

## 5 · Leakage-safe split

`StratifiedGroupKFold` holds out about 14 % of the **groups** as a final `test` set. Everything else (`dev`) drives
statistics, model selection and the optimizer study.

In [ ]:
y_all = df.label.to_numpy(dtype=object)
groups_all = df.group.to_numpy()
outer = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
dev_idx, test_idx = next(outer.split(np.zeros(len(df)), y_all, groups_all))
df["split"] = "dev"
df.loc[test_idx, "split"] = "test"
assert not set(groups_all[dev_idx]) & set(groups_all[test_idx]), "a group crosses dev/test"

naive_dev, naive_test = train_test_split(np.arange(len(df)), test_size=len(test_idx), stratify=y_all, random_state=SEED)
leak = pd.Series(np.isin(groups_all[naive_test], groups_all[naive_dev]), index=naive_test)
print("naive random split - share of test messages with a near-copy in train:",
      {c: f"{leak[y_all[naive_test] == c].mean():.1%}" for c in CLASSES}, "| grouped split: 0% (asserted)")
split_table = pd.crosstab(df.split, df.label).reindex(columns=CLASSES)
split_table["total"] = split_table.sum(axis=1)
split_table

## 6 · Features

* `model_text` = `normalize_for_model(message)`: lower-case, entity type tokens (`phonetoken`, `urltoken`, …).
* The 15 cue flags and 11 style features from the shared pipeline. The three **case-dependent** ones are excluded
  from every model and test here (§2.2).

In [ ]:
df["model_text"] = df.message.map(sp.normalize_for_model)
feats = pd.DataFrame([sp.feature_row(m) for m in df.message])
df = pd.concat([df.drop(columns=[c for c in sp.NUM_COLS if c in df], errors="ignore"), feats], axis=1)
CASE_DEPENDENT = ["upper_ratio", "caps_word_count", "cue_shouting"]
SAFE_CONT = [c for c in sp.CONT_COLS if c not in CASE_DEPENDENT]
SAFE_CUES = [c for c in sp.CUE_COLS if c not in CASE_DEPENDENT]
print("median upper_ratio by class (the artifact):", df.groupby("label").upper_ratio.median().round(3).to_dict())

dev = df[df.split == "dev"].reset_index(drop=True)
test = df[df.split == "test"].reset_index(drop=True)
y_dev, g_dev = dev.label.to_numpy(dtype=object), dev.group.to_numpy()
y_test, g_test = test.label.to_numpy(dtype=object), test.group.to_numpy()
print(f"dev: {len(dev):,} messages in {len(set(g_dev)):,} groups | test: {len(test):,} messages in {len(set(g_test)):,} groups")

## 7 · Statistical analysis (dev only)

| Question | Method | Why |
|---|---|---|
| Do continuous features differ across the 3 classes? | **Kruskal–Wallis H** + η²(H) | non-parametric k-sample test; the features are heavily skewed |
| Do they separate *spam from smishing* specifically? | **Mann–Whitney U** + Cliff's δ | the hard boundary is spam vs smishing, not ham vs the rest |
| Is each evidence signal associated with the label? | **χ² (3×2)** + **Cramér's V**; spam-vs-smishing **odds ratio** | signal–label consistency (project plan §4) |
| Many tests? | Benjamini–Hochberg FDR | — |
| Which words separate smishing from spam? | weighted log-odds, informative Dirichlet prior | variance-aware word ranking |
| How noisy are the labels? | out-of-fold probabilities, confident-learning style | finds rows the data itself contradicts |

In [ ]:
eff = dev.groupby("label").agg(messages=("message", "size"), template_groups=("group", "nunique")).reindex(CLASSES)
eff["share of messages %"] = (100 * eff.messages / eff.messages.sum()).round(1)
eff["share of groups %"] = (100 * eff.template_groups / eff.template_groups.sum()).round(1)
display(eff)

fig, ax = plt.subplots(figsize=(9, 4))
for lab in CLASSES:
    v = np.sort(dev.loc[dev.label == lab, "char_len"].to_numpy())
    ax.step(v, np.arange(1, len(v) + 1) / len(v), where="post", color=CLASS_COLORS[lab], label=lab)
ax.axvline(160, color=INK_2, lw=1)
ax.text(165, 0.05, "160 = one SMS", color=INK_2, fontsize=9)
ax.set_xlim(0, 400)
ax.set_xlabel("characters")
ax.set_ylabel("cumulative share of messages")
ax.legend(loc="lower right")
finish(ax, "Spam and smishing both fill the SMS; ham does not", "ECDF of message length, dev set")
plt.show()

### 7.1 · Continuous features: 3-class differences and the spam-vs-smishing boundary

In [ ]:
def cliffs_delta(a, b):
    u = stats.mannwhitneyu(a, b, alternative="two-sided").statistic
    return 2 * u / (len(a) * len(b)) - 1


rows = []
for col in SAFE_CONT:
    samples = [dev.loc[dev.label == c, col].to_numpy(float) for c in CLASSES]
    try:
        kw = stats.kruskal(*samples)
        h, kw_p = kw.statistic, kw.pvalue
    except ValueError:                       # all values identical
        h, kw_p = 0.0, 1.0
    smish, spam_ = samples[1], samples[2]
    rows.append({"feature": col, **{f"median {c}": float(np.median(s)) for c, s in zip(CLASSES, samples)},
                 "KW H": h, "KW p": kw_p, "eta²(H)": (h - len(CLASSES) + 1) / (len(dev) - len(CLASSES)),
                 "δ smishing vs spam": cliffs_delta(smish, spam_),
                 "MW p (smishing vs spam)": stats.mannwhitneyu(smish, spam_, alternative="two-sided").pvalue})
cont_table = pd.DataFrame(rows)
cont_table["KW q"] = stats.false_discovery_control(cont_table["KW p"].to_numpy())
cont_table["MW q"] = stats.false_discovery_control(cont_table["MW p (smishing vs spam)"].to_numpy())
cont_table = cont_table.sort_values("eta²(H)", ascending=False).drop(columns=["KW p", "MW p (smishing vs spam)"])
cont_table.style.format({"KW H": "{:.0f}", "eta²(H)": "{:.3f}", "δ smishing vs spam": "{:+.3f}", "KW q": "{:.1e}",
                         "MW q": "{:.1e}", **{f"median {c}": "{:.3g}" for c in CLASSES}})

### 7.2 · Evidence signals: consistency with the labels

The project plan asks for a check that the signals are consistent with the labels. For each signal the table shows:
* the rate in each class;
* Cramér's V over all three classes;
* the **odds ratio for smishing vs spam**, which says what *defines* smishing in this dataset.

In [ ]:
rows = []
for col in SAFE_CUES:
    rates = {c: 100 * dev.loc[dev.label == c, col].mean() for c in CLASSES}
    table = pd.crosstab(dev.label, dev[col]).reindex(index=CLASSES, columns=[0, 1], fill_value=0).to_numpy()
    if (table.sum(axis=0) > 0).all():
        chi = stats.chi2_contingency(table)
        chi_stat, chi_p = chi.statistic, chi.pvalue
    else:
        chi_stat, chi_p = 0.0, 1.0
    a = int(((dev.label == "smishing") & (dev[col] == 1)).sum()); b = int(((dev.label == "spam") & (dev[col] == 1)).sum())
    c_ = int(((dev.label == "smishing") & (dev[col] == 0)).sum()); d = int(((dev.label == "spam") & (dev[col] == 0)).sum())
    log_or = math.log((a + 0.5) * (d + 0.5) / ((b + 0.5) * (c_ + 0.5)))
    se = math.sqrt(1 / (a + 0.5) + 1 / (b + 0.5) + 1 / (c_ + 0.5) + 1 / (d + 0.5))
    rows.append({"signal": col.removeprefix("cue_"), **{f"% {c}": r for c, r in rates.items()},
                 "Cramér's V": math.sqrt(chi_stat / table.sum()), "chi2 p": chi_p,
                 "OR smishing:spam": math.exp(log_or), "OR low": math.exp(log_or - 1.96 * se),
                 "OR high": math.exp(log_or + 1.96 * se)})
signal_table = pd.DataFrame(rows)
signal_table["q (BH)"] = stats.false_discovery_control(signal_table["chi2 p"].to_numpy())
signal_table = signal_table.sort_values("Cramér's V", ascending=False).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(9.5, 5.4))
order = signal_table.iloc[::-1].reset_index(drop=True)
for c in CLASSES:
    ax.scatter(order[f"% {c}"], range(len(order)), s=46, color=CLASS_COLORS[c], edgecolor=SURFACE, linewidth=2, zorder=3)
ax.set_yticks(range(len(order)), order.signal)
ax.set_xlabel("% of messages in the class with the signal")
class_legend(ax)
finish(ax, "Signal rates by class", "dev set · rows sorted by Cramér's V (association with the 3-class label)")
plt.tight_layout()
plt.show()
signal_table.style.format({**{f"% {c}": "{:.1f}" for c in CLASSES}, "Cramér's V": "{:.2f}", "chi2 p": "{:.1e}",
                           "OR smishing:spam": "{:.2f}", "OR low": "{:.2f}", "OR high": "{:.2f}", "q (BH)": "{:.1e}"})

**Reading.**
* *Smishing* here leans towards **phone numbers, prize/reward language, urgency, account/OTP words and generic
  greetings**: "call this number to claim", "your account will be blocked".
* *Spam* leans towards **short codes, "free", opt-out wording and premium charges**: marketing and subscriptions.

That is a coherent distinction, but with heavy overlap, and section 7.4 shows the labels themselves disagree on many
messages. Signals near 0 % in ham, such as `phone` and `short_code`, are partly the UCI anonymisation artifact
(notebook 02).

### 7.3 · Words that separate smishing from spam

In [ ]:
def log_odds_dirichlet(texts_a, texts_b, min_df=3):
    cv = CountVectorizer(token_pattern=r"(?u)\b\w+\b", min_df=min_df)
    X = cv.fit_transform(list(texts_a) + list(texts_b))
    ya, yb = np.asarray(X[:len(texts_a)].sum(0)).ravel(), np.asarray(X[len(texts_a):].sum(0)).ravel()
    prior = ya + yb
    a0, na, nb = prior.sum(), ya.sum(), yb.sum()
    delta = np.log((ya + prior) / (na + a0 - ya - prior)) - np.log((yb + prior) / (nb + a0 - yb - prior))
    return pd.Series(delta / np.sqrt(1 / (ya + prior) + 1 / (yb + prior)), index=cv.get_feature_names_out()).sort_values()


z = log_odds_dirichlet(dev.model_text[dev.label == "smishing"], dev.model_text[dev.label == "spam"])
top = pd.concat([z.head(15), z.tail(15)])
fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(range(len(top)), top.to_numpy(), color=np.where(top > 0, SMISH_C, SPAM_C), height=0.7)
ax.set_yticks(range(len(top)), top.index, fontsize=9)
ax.axvline(0, color=AXIS, lw=1)
ax.set_xlabel("log-odds z-score (positive: smishing-leaning)")
ax.legend(handles=[Patch(color=SMISH_C, label="smishing-leaning"), Patch(color=SPAM_C, label="spam-leaning")], loc="lower right")
finish(ax, "What the labels mean: smishing vs spam vocabulary", "weighted log-odds with informative Dirichlet prior, dev set")
plt.tight_layout()
plt.show()
if fingerprint is not None:
    fp_words = set(fingerprint["swapped in"].head(10))
    print("augmentation words among the 30 bars above:", sorted(fp_words & set(top.index)) or "none")

### 7.4 · Label noise: where does the data contradict itself?

Out-of-fold probabilities come from a grouped 5-fold TF-IDF + logistic regression. A message is *suspect* when the
model gives its **given** label a probability below 0.1. This is the core idea of confident learning
(Northcutt et al., 2021): it flags rows to **audit**, not rows to delete automatically.

In [ ]:
def make_word_tfidf():
    return TfidfVectorizer(analyzer="word", token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2), min_df=2,
                           sublinear_tf=True, dtype=np.float32)


def make_char_tfidf():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, dtype=np.float32)


def tfidf_blocks(train_texts, test_texts):
    word, char = make_word_tfidf(), make_char_tfidf()
    X_tr = sparse.hstack([word.fit_transform(train_texts), char.fit_transform(train_texts)]).tocsr()
    X_te = sparse.hstack([word.transform(test_texts), char.transform(test_texts)]).tocsr()
    return X_tr, X_te


oof = np.zeros((len(dev), len(CLASSES)))
for tr, te in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(dev, y_dev, g_dev):
    X_tr, X_te = tfidf_blocks(dev.model_text.iloc[tr], dev.model_text.iloc[te])
    oof[te] = LogisticRegression(C=1.0, max_iter=3000).fit(X_tr, y_dev[tr]).predict_proba(X_te)

given_idx = np.array([CLASSES.index(c) for c in y_dev])
p_given = oof[np.arange(len(dev)), given_idx]
predicted = np.array(CLASSES, dtype=object)[oof.argmax(axis=1)]
suspect = p_given < 0.1
print(f"suspect labels: {suspect.sum()} of {len(dev)} ({suspect.mean():.1%})")
display(pd.crosstab(pd.Series(y_dev[suspect], name="given"), pd.Series(predicted[suspect], name="model says")))
dev.assign(p_given=p_given, model_says=predicted)[suspect].sort_values("p_given")[["label", "model_says", "p_given", "message"]].head(12)

## 8 · Which models to try: grouped, repeated cross-validation

| Family | Representation | Why it is a candidate |
|---|---|---|
| Majority class | — | floor |
| Complement NB | word 1–2-gram counts | classic text baseline; CNB is built for uneven classes; very fast |
| Complement NB **+ calibration** | same | NB ranks well but its probabilities are over-confident; sigmoid (Platt) calibration fixes that |
| TF-IDF **logistic regression** (multinomial, L-BFGS) | word 1–2 + char_wb 2–5 TF-IDF | convex, calibrated by construction, additive token explanations, KB-sized |
| TF-IDF **linear SVM** | same | strong margin classifier; no probabilities |
| TF-IDF linear SVM **+ calibration** | same | SVM ranking + Platt probabilities |
| TF-IDF **SGD** (log-loss, averaged) | same | same model family as LR, trained online: the candidate for **on-device incremental updates** |
| **MLP** (Adam) | LSA-300 (TruncatedSVD of TF-IDF) | does a non-linear model find interactions the linear ones miss? |
| **Gradient boosting** (HistGB) | LSA-100 + case-free cues/style | strongest tabular learner; tests hand-made signals + dense text summary |

**Protocol.**
* `StratifiedGroupKFold` with `N_FOLDS` × `N_REPEATS` on `dev`.
* Every vectoriser, SVD, scaler and calibrator is fit **inside** the training fold. Calibration uses a *grouped*
  inner split, so a calibrator never sees a paraphrase of its training data.

**Metrics.**
* Macro-F1 (3-class) and its **group-weighted** variant.
* Per-class F1.
* *Subtype* macro-F1 (smishing vs spam on truly malicious rows).
* Ham-vs-malicious PR-AUC and recall at ≤ 1 % ham false alarms.
* Log-loss (calibration) and fit time.

**Selection rule, fixed before the run:**
1. The leader is the family with the highest macro-F1.
2. Families not significantly worse than the leader on macro-F1 **and** on recall at 1 % FPR count as tied. The test
   is the Nadeau–Bengio corrected resampled t-test, BH-adjusted, q ≥ 0.05.
3. Among the tied families that output probabilities, ship the **lowest log-loss**. Remaining ties (Δ < 0.001) go
   to the simplest.

In [ ]:
def fold_features(train_df, test_df):
    X_tr, X_te = tfidf_blocks(train_df.model_text, test_df.model_text)
    counts = CountVectorizer(token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 2), min_df=2)
    K_tr, K_te = counts.fit_transform(train_df.model_text), counts.transform(test_df.model_text)
    svd = TruncatedSVD(n_components=N_SVD, random_state=SEED).fit(X_tr)
    lsa_scaler = StandardScaler().fit(svd.transform(X_tr))
    Z_tr, Z_te = lsa_scaler.transform(svd.transform(X_tr)), lsa_scaler.transform(svd.transform(X_te))
    cue_scaler = StandardScaler().fit(train_df[SAFE_CONT + SAFE_CUES].to_numpy(float))
    C_tr = cue_scaler.transform(train_df[SAFE_CONT + SAFE_CUES].to_numpy(float))
    C_te = cue_scaler.transform(test_df[SAFE_CONT + SAFE_CUES].to_numpy(float))
    return {"tfidf": (X_tr, X_te), "counts": (K_tr, K_te), "lsa": (Z_tr, Z_te),
            "lsa+cues": (np.hstack([Z_tr[:, :100], C_tr]), np.hstack([Z_te[:, :100], C_te]))}


FAMILY_BLOCK = {
    "Majority class": "counts",
    "Complement NB": "counts",
    "Complement NB + calibration": "counts",
    "TF-IDF LR": "tfidf",
    "TF-IDF Linear SVM": "tfidf",
    "TF-IDF Linear SVM + calibration": "tfidf",
    "TF-IDF SGD (log-loss, averaged)": "tfidf",
    "LSA MLP (Adam)": "lsa",
    "LSA + cues gradient boosting": "lsa+cues",
}
GRID = {
    "Majority class": [{}],
    "Complement NB": [{"alpha": a} for a in (0.3, 1.0, 3.0)],
    "Complement NB + calibration": [{"alpha": a} for a in (0.3, 1.0, 3.0)],
    "TF-IDF LR": [{"C": C} for C in (0.3, 1.0, 3.0, 10.0)],
    "TF-IDF Linear SVM": [{"C": C} for C in (0.03, 0.1, 0.3)],
    "TF-IDF Linear SVM + calibration": [{"C": C} for C in (0.1, 0.3)],
    "TF-IDF SGD (log-loss, averaged)": [{"alpha": a} for a in (1e-5, 3e-5, 1e-4)],
    "LSA MLP (Adam)": [{"alpha": a} for a in (1e-3, 1e-2)],
    "LSA + cues gradient boosting": [{}],
}
SIMPLICITY = {"Majority class": 0, "Complement NB": 1, "Complement NB + calibration": 2, "TF-IDF LR": 3,
              "TF-IDF SGD (log-loss, averaged)": 3, "TF-IDF Linear SVM": 3, "TF-IDF Linear SVM + calibration": 4,
              "LSA MLP (Adam)": 5, "LSA + cues gradient boosting": 5}


def make_estimator(family, params, inner_cv=None):
    if family == "Majority class":
        return DummyClassifier(strategy="prior")
    if family == "Complement NB":
        return ComplementNB(**params)
    if family == "Complement NB + calibration":
        return CalibratedClassifierCV(ComplementNB(**params), method="sigmoid", cv=inner_cv)
    if family == "TF-IDF LR":
        return LogisticRegression(max_iter=3000, **params)
    if family == "TF-IDF Linear SVM":
        return LinearSVC(**params)
    if family == "TF-IDF Linear SVM + calibration":
        return CalibratedClassifierCV(LinearSVC(**params), method="sigmoid", cv=inner_cv)
    if family == "TF-IDF SGD (log-loss, averaged)":
        return SGDClassifier(loss="log_loss", learning_rate="adaptive", eta0=0.1, average=True, max_iter=50, tol=1e-4,
                             random_state=SEED, **params)
    if family == "LSA MLP (Adam)":
        return MLPClassifier(hidden_layer_sizes=(128,), solver="adam", learning_rate_init=1e-3, batch_size=64,
                             max_iter=MLP_EPOCHS, random_state=SEED, **params)
    if family == "LSA + cues gradient boosting":
        return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1, early_stopping=True, random_state=SEED)
    raise ValueError(family)


def malicious_score(model, X):
    classes = list(model.classes_)
    if hasattr(model, "predict_proba"):
        P = model.predict_proba(X)
        return P, 1.0 - P[:, classes.index("ham")]
    D = model.decision_function(X)
    return None, -D[:, classes.index("ham")]


def recall_at_fpr(y_bin, score, max_fpr=0.01):
    fpr, tpr, _ = roc_curve(y_bin, score)
    return float(tpr[fpr <= max_fpr].max())


def evaluate(model, X, y, weights):
    yhat = model.predict(X)
    P, mal = malicious_score(model, X)
    y_bin = (y != "ham").astype(int)
    is_mal = y != "ham"
    per_class = f1_score(y, yhat, labels=CLASSES, average=None, zero_division=0)
    return {
        "macro-F1": f1_score(y, yhat, labels=CLASSES, average="macro", zero_division=0),
        "group-weighted macro-F1": f1_score(y, yhat, labels=CLASSES, average="macro", sample_weight=weights, zero_division=0),
        **{f"F1 {c}": v for c, v in zip(CLASSES, per_class)},
        "subtype macro-F1": f1_score(y[is_mal], yhat[is_mal], labels=["smishing", "spam"], average="macro", zero_division=0),
        "malicious PR-AUC": average_precision_score(y_bin, mal),
        "recall@ham-FPR≤1%": recall_at_fpr(y_bin, mal),
        "log-loss": log_loss(y, P, labels=list(model.classes_)) if P is not None else np.nan,
    }


records, framing, t_start = [], [], time.time()
for rep in range(N_REPEATS):
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED + rep)
    for fold, (tr, te) in enumerate(cv.split(dev, y_dev, g_dev)):
        blocks = fold_features(dev.iloc[tr], dev.iloc[te])
        y_tr, y_te, g_tr, g_te = y_dev[tr], y_dev[te], g_dev[tr], g_dev[te]
        inner_cv = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(np.zeros(len(tr)), y_tr, g_tr))
        w_te = 1.0 / pd.Series(g_te).map(pd.Series(g_te).value_counts()).to_numpy()
        for family, grid in GRID.items():
            X_tr, X_te = blocks[FAMILY_BLOCK[family]]
            for params in grid:
                t = time.time()
                model = make_estimator(family, params, inner_cv).fit(X_tr, y_tr)
                fit_s = time.time() - t
                records.append({"repeat": rep, "fold": fold, "family": family, "config": json.dumps(params),
                                "n_train": len(tr), "n_test": len(te), "fit s": fit_s,
                                **evaluate(model, X_te, y_te, w_te)})
        # task framing: one binary detector + one subtype model, vs the 3-class model
        X_tr, X_te = blocks["tfidf"]
        yb_tr, yb_te = (y_tr != "ham").astype(int), (y_te != "ham").astype(int)
        for C in (1.0, 10.0):
            s = LogisticRegression(C=C, max_iter=3000).fit(X_tr, yb_tr).predict_proba(X_te)[:, 1]
            mal_tr, mal_te = y_tr != "ham", y_te != "ham"
            sub = LogisticRegression(C=C, max_iter=3000).fit(X_tr[mal_tr], y_tr[mal_tr]).predict(X_te[mal_te])
            framing.append({"repeat": rep, "fold": fold, "C": C, "malicious PR-AUC": average_precision_score(yb_te, s),
                            "recall@ham-FPR≤1%": recall_at_fpr(yb_te, s),
                            "subtype macro-F1": f1_score(y_te[mal_te], sub, average="macro")})
        print(f"repeat {rep} fold {fold} done ({time.time() - t_start:.0f}s)", flush=True)

cv_results = pd.DataFrame(records)
METRICS = ["macro-F1", "group-weighted macro-F1", "F1 ham", "F1 smishing", "F1 spam", "subtype macro-F1",
           "malicious PR-AUC", "recall@ham-FPR≤1%", "log-loss", "fit s"]
best_config = (cv_results.groupby(["family", "config"])["macro-F1"].mean().reset_index()
               .sort_values("macro-F1").groupby("family").tail(1).set_index("family")["config"])
best = cv_results[cv_results.apply(lambda r: best_config[r.family] == r.config, axis=1)]
summary = best.groupby("family")[METRICS].agg(["mean", "std"])
summary.insert(0, ("config", ""), best_config)
summary = summary.sort_values(("macro-F1", "mean"), ascending=False)
summary.style.format(precision=3, na_rep="n/a")

In [ ]:
grid_view = cv_results.groupby(["family", "config"])[["macro-F1", "group-weighted macro-F1", "log-loss"]].mean()
display(grid_view.style.format("{:.4f}", na_rep="n/a"))

framing_view = pd.DataFrame(framing).groupby("C")[["malicious PR-AUC", "recall@ham-FPR≤1%", "subtype macro-F1"]].mean()
three_class = summary.loc["TF-IDF LR", [("malicious PR-AUC", "mean"), ("recall@ham-FPR≤1%", "mean"), ("subtype macro-F1", "mean")]]
framing_view.loc["3-class TF-IDF LR (best C)"] = three_class.to_numpy()
print("Task framing - two-stage (binary detector + subtype model) vs one 3-class model:")
framing_view.round(4)

In [ ]:
def corrected_resampled_ttest(a, b, n_train, n_test):
    # Nadeau & Bengio (2003): paired t-test over CV folds, variance inflated by n_test / n_train.
    d = np.asarray(a) - np.asarray(b)
    var = d.var(ddof=1)
    if var == 0:
        return 0.0, 1.0
    t = d.mean() / math.sqrt((1 / len(d) + n_test / n_train) * var)
    return t, 2 * stats.t.sf(abs(t), df=len(d) - 1)


LEADER = summary.index[0]
lead = best[best.family == LEADER].sort_values(["repeat", "fold"])
n_tr, n_te = lead.n_train.mean(), lead.n_test.mean()
rows = []
for family in summary.index.drop(LEADER):
    other = best[best.family == family].sort_values(["repeat", "fold"])
    for metric in ("macro-F1", "recall@ham-FPR≤1%"):
        t, p = corrected_resampled_ttest(lead[metric].to_numpy(), other[metric].to_numpy(), n_tr, n_te)
        rows.append({"family": family, "metric": metric, f"{LEADER} − family": lead[metric].mean() - other[metric].mean(),
                     "t": t, "p (corrected)": p})
comparison = pd.DataFrame(rows)
comparison["q (BH)"] = stats.false_discovery_control(comparison["p (corrected)"].to_numpy())

tied = [LEADER] + [f for f, part in comparison.groupby("family") if (part["q (BH)"] >= 0.05).all()]
probabilistic = [f for f in tied if not np.isnan(summary.loc[f, ("log-loss", "mean")])]
if not probabilistic:      # e.g. only the plain SVM is tied: fall back to the best probabilistic family overall
    probabilistic = [f for f in summary.index if not np.isnan(summary.loc[f, ("log-loss", "mean")]) and f != "Majority class"][:1]
CHOSEN = min(probabilistic, key=lambda f: (round(summary.loc[f, ("log-loss", "mean")], 3), SIMPLICITY[f]))
CHOSEN_PARAMS = json.loads(best_config[CHOSEN])

fig, ax = plt.subplots(figsize=(9.5, 4.4))
fams = list(summary.index[::-1])
for i, fam in enumerate(fams):
    vals = best.loc[best.family == fam, "macro-F1"].to_numpy()
    color = STAT_C if fam == CHOSEN else MUTED
    ax.scatter(vals, np.full(len(vals), i) + RNG.uniform(-0.12, 0.12, len(vals)), s=16, color=color, alpha=0.7, linewidths=0)
    ax.scatter(vals.mean(), i, s=90, color=color, edgecolor=SURFACE, linewidth=2, zorder=3)
ax.set_yticks(range(len(fams)), fams)
ax.set_xlim(0.75, 0.95)
ax.set_xlabel("macro-F1 per CV fold (large dot = mean; the majority class sits off-scale)")
finish(ax, "Fold-level macro-F1, best configuration per family", f"grouped CV on dev · violet = selected ({CHOSEN})")
plt.tight_layout()
plt.show()
display(comparison.style.format({f"{LEADER} − family": "{:+.4f}", "t": "{:.2f}", "p (corrected)": "{:.3g}", "q (BH)": "{:.3g}"}))
print(f"leader (macro-F1)        : {LEADER}")
print(f"statistically tied       : {tied}")
print(f"tied + probabilistic     : {probabilistic}")
print(f"-> SELECTED              : {CHOSEN} {CHOSEN_PARAMS} (CV log-loss {summary.loc[CHOSEN, ('log-loss', 'mean')]:.4f})")

**How to read the model comparison**
* **Macro-F1 vs group-weighted macro-F1.** If a family drops much more on the group-weighted metric, it is
  exploiting augmentation templates.
* **The F1 per class** shows where the errors are. Expect ham ≈ 0.97 and the spam/smishing boundary well below
  that. The **subtype macro-F1** isolates the spam-vs-smishing decision.
* **Task framing table.** If a dedicated binary detector matches the 3-class model on ham-vs-malicious metrics, a
  *two-stage* design is attractive:
  * a high-precision malicious detector, which decides whether to warn;
  * a subtype model, which decides which advice card to show.

  Then the noisy spam/smishing boundary can never cause a missed warning.
* **Calibration.** In the prototype, NB had the best macro-F1 but the worst log-loss. If NB or SVM lead on F1, the
  calibrated variants are how you keep that ranking and still get usable confidences.

## 9 · Which optimizers work best, and why

Optimizer choice means different things for different models:

| Model type | Loss surface | What the optimizer decides |
|---|---|---|
| Logistic regression | **convex**: one global optimum | only *speed* and *numerical robustness*; every converged solver gives the same model |
| SGD-trained linear model | convex, but optimised stochastically | speed, and how close you get without full passes (matters for on-device updates) |
| MLP / transformer | **non-convex** | *which* solution you reach, how fast, and how sensitive training is to the learning rate; regularisation interacts with the optimizer (Adam + L2 vs AdamW) |

All studies use one grouped train/validation split inside `dev`; test is untouched.

In [ ]:
opt_tr, opt_va = next(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(dev, y_dev, g_dev))
X_otr, X_ova = tfidf_blocks(dev.model_text.iloc[opt_tr], dev.model_text.iloc[opt_va])
y_otr, y_ova = y_dev[opt_tr], y_dev[opt_va]
C_BEST = json.loads(best_config["TF-IDF LR"])["C"]
print(f"optimizer study split: {len(opt_tr):,} train / {len(opt_va):,} validation messages; LR C = {C_BEST}")

### 9.1 · Convex case: logistic-regression solvers

| Solver | Method | Expect |
|---|---|---|
| `lbfgs` | quasi-Newton with a limited-memory curvature estimate | fast and robust; the default |
| `newton-cg` | truncated Newton (CG on Hessian-vector products) | fewest iterations; cheap here because Hessian-vector products are cheap on sparse data |
| `sag` | stochastic average gradient | linear convergence, but needs many passes when regularisation is weak |
| `saga` | SAG + proximal step | the only one supporting L1/elastic-net, and the slowest here |
| `liblinear` | coordinate descent | **binary only**; for 3 classes it needs one-vs-rest, which is a *different* model |

In [ ]:
SOLVERS = {
    "lbfgs": LogisticRegression(C=C_BEST, solver="lbfgs", max_iter=1000),
    "newton-cg": LogisticRegression(C=C_BEST, solver="newton-cg", max_iter=1000),
    "sag": LogisticRegression(C=C_BEST, solver="sag", max_iter=1000),
    "saga": LogisticRegression(C=C_BEST, solver="saga", max_iter=1000),
    "liblinear (one-vs-rest)": OneVsRestClassifier(LogisticRegression(C=C_BEST, solver="liblinear", max_iter=1000)),
}
solver_rows, P_ref = [], None
for name, est in SOLVERS.items():
    t = time.time()
    est.fit(X_otr, y_otr)
    secs = time.time() - t
    P = est.predict_proba(X_ova)
    P_ref = P if P_ref is None else P_ref
    iters = int(np.max(est.n_iter_)) if hasattr(est, "n_iter_") else int(max(np.max(e.n_iter_) for e in est.estimators_))
    solver_rows.append({"solver": name, "fit s": secs, "iterations": iters,
                        "val log-loss": log_loss(y_ova, P, labels=CLASSES),
                        "val macro-F1": f1_score(y_ova, est.predict(X_ova), average="macro"),
                        "max |Δp| vs lbfgs": float(np.abs(P - P_ref).max())})
solver_table = pd.DataFrame(solver_rows).set_index("solver")

fig, ax = plt.subplots(figsize=(8.5, 3.2))
ax.barh(range(len(solver_table)), solver_table["fit s"], color=STAT_C, height=0.55)
for i, (secs, it) in enumerate(zip(solver_table["fit s"], solver_table.iterations)):
    ax.text(secs * 1.08, i, f"{secs:.1f}s · {it} it", va="center", fontsize=9, color=INK_2)
ax.set_yticks(range(len(solver_table)), solver_table.index)
ax.set_xscale("log")
ax.invert_yaxis()
ax.set_xlabel("fit time (s, log scale)")
finish(ax, "Same convex problem, very different cost", f"multinomial LR, C = {C_BEST}, tol = 1e-4")
plt.tight_layout()
plt.show()
solver_table.style.format({"fit s": "{:.2f}", "val log-loss": "{:.4f}", "val macro-F1": "{:.4f}", "max |Δp| vs lbfgs": "{:.1e}"})

**Reading.** The four multinomial solvers agree to within numerical tolerance (`max |Δp|` is tiny), as convexity
guarantees. So **for logistic regression the "best optimizer" is simply the fastest reliable one: L-BFGS or
newton-cg**.
* `sag`/`saga` pay off only on much larger *n*, or when you need L1/elastic-net (`saga`). They slow down sharply when
  regularisation is weak: in the prototype, at C = 30, tol 1e-6, they took 134 s and 239 s against 2–3 s.
* `liblinear` changes the model to one-vs-rest. Its probabilities are renormalised binary scores, not a joint
  softmax.

### 9.2 · Stochastic linear training: SGD learning-rate schedules

Same model (multinomial-style log-loss), same L2 strength: `alpha = 1 / (C·n)` matches the LR objective. It is
trained one epoch at a time with `partial_fit`, which is exactly how an app would apply **local feedback updates**
(the "Report / Dismiss" step of the user journey).
* `optimal`: η = 1/(α(t + t₀)) (Bottou)
* `constant`: η = η₀
* `invscaling`: η = η₀ / t^0.5
* each with and without **iterate averaging** (Polyak–Ruppert / ASGD)

(`adaptive` behaves like `constant` under `partial_fit`, because its step-down rule only runs inside `fit`.)

In [ ]:
alpha_matched = 1.0 / (C_BEST * len(opt_tr))
SCHEDULES = {"optimal": {}, "constant": {"eta0": 0.1}, "invscaling": {"eta0": 0.5, "power_t": 0.5}}
ref_loss = solver_table.loc["lbfgs", "val log-loss"]
sgd_curves = {}
for sched, kw in SCHEDULES.items():
    for avg in (False, True):
        clf = SGDClassifier(loss="log_loss", alpha=alpha_matched, learning_rate=sched, average=avg, random_state=SEED, **kw)
        losses = []
        for _ in range(OPT_EPOCHS):
            clf.partial_fit(X_otr, y_otr, classes=np.array(CLASSES, dtype=object))
            losses.append(log_loss(y_ova, clf.predict_proba(X_ova), labels=CLASSES))
        sgd_curves[(sched, avg)] = losses

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), sharey=True)
for ax, sched in zip(axes, SCHEDULES):
    ep = np.arange(1, OPT_EPOCHS + 1)
    ax.plot(ep, sgd_curves[(sched, False)], color=STAT_C, label="plain SGD")
    ax.plot(ep, sgd_curves[(sched, True)], color=INK_2, label="averaged (ASGD)")
    ax.axhline(ref_loss, color=AXIS, lw=1)
    ax.text(OPT_EPOCHS, ref_loss, " L-BFGS", va="bottom", ha="right", fontsize=8, color=MUTED)
    ax.set_xlabel("epoch")
    finish(ax, sched)
axes[0].set_ylabel("validation log-loss")
axes[0].set_ylim(0, max(1.0, ref_loss * 2))
axes[0].legend(loc="upper right")
plt.tight_layout()
plt.show()
pd.DataFrame({f"{s} {'avg' if a else 'plain'}": [v[0], v[min(4, len(v) - 1)], v[-1]] for (s, a), v in sgd_curves.items()},
             index=["epoch 1", "epoch 5", f"epoch {OPT_EPOCHS}"]).T.round(4)

**Reading.**
* A constant step (or `optimal`) reaches the L-BFGS solution's quality in a handful of epochs. It can even dip below
  it, because stopping early acts as extra regularisation.
* `invscaling` decays too fast and stalls.
* **Averaging** smooths the noisy final iterate. That makes it the safer choice for *unattended* on-device updates,
  where nobody watches a validation curve.

**Recommendation for on-device feedback:** `SGDClassifier(loss="log_loss", learning_rate="constant", average=True)`
warm-started from the L-BFGS weights, with a small `eta0`.

### 9.3 · Non-convex case: SGD, momentum, Nesterov, AdaGrad, RMSprop, Adam, AdamW on an MLP

scikit-learn's MLP only offers `sgd`, `adam` and `lbfgs`. To compare every optimizer you would meet in PyTorch, we
use a minimal NumPy MLP: 300-d LSA input → 128 ReLU → 3-way softmax, minibatch 64. Its optimizers implement the same
update rules as `torch.optim`. A finite-difference **gradient check** proves the backward pass is right before any
result is trusted.

In [ ]:
class TinyMLP:
    # One hidden ReLU layer + softmax cross-entropy; parameters in a dict so optimizers are pluggable.
    def __init__(self, n_in, n_hidden, n_out, seed):
        rng = np.random.default_rng(seed)
        self.p = {"W1": rng.normal(0, math.sqrt(2 / n_in), (n_in, n_hidden)), "b1": np.zeros(n_hidden),
                  "W2": rng.normal(0, math.sqrt(1 / n_hidden), (n_hidden, n_out)), "b2": np.zeros(n_out)}

    def forward(self, X):
        z1 = X @ self.p["W1"] + self.p["b1"]
        h = np.maximum(z1, 0)
        logits = h @ self.p["W2"] + self.p["b2"]
        logits = logits - logits.max(axis=1, keepdims=True)
        P = np.exp(logits)
        return z1, h, P / P.sum(axis=1, keepdims=True)

    def loss_and_grads(self, X, Y):
        z1, h, P = self.forward(X)
        n = len(X)
        loss = -np.log(np.clip(P[np.arange(n), Y], 1e-12, None)).mean()
        d = P.copy()
        d[np.arange(n), Y] -= 1
        d /= n
        g = {"W2": h.T @ d, "b2": d.sum(0)}
        dh = d @ self.p["W2"].T
        dh[z1 <= 0] = 0
        g["W1"], g["b1"] = X.T @ dh, dh.sum(0)
        return loss, g

    def predict_proba(self, X):
        return self.forward(X)[2]


class Optimizer:
    # Update rules follow torch.optim: SGD, SGD+momentum, Nesterov, Adagrad, RMSprop, Adam, AdamW.
    def __init__(self, kind, lr, weight_decay=0.0, beta1=0.9, beta2=0.999, eps=1e-8, momentum=0.9, rho=0.9):
        self.kind, self.lr, self.wd = kind, lr, weight_decay
        self.b1, self.b2, self.eps, self.mu, self.rho = beta1, beta2, eps, momentum, rho
        self.state, self.t = {}, 0

    def step(self, params, grads):
        self.t += 1
        for k, w in params.items():
            g = grads[k]
            decay = self.wd if k.startswith("W") else 0.0          # no decay on biases
            if self.kind != "adamw" and decay:
                g = g + decay * w                                    # coupled L2: decay enters the gradient
            s = self.state.setdefault(k, {})
            if self.kind == "sgd":
                w -= self.lr * g
            elif self.kind in ("momentum", "nesterov"):
                v = self.mu * s.get("v", np.zeros_like(w)) + g
                s["v"] = v
                w -= self.lr * ((g + self.mu * v) if self.kind == "nesterov" else v)
            elif self.kind == "adagrad":
                G = s.get("G", np.zeros_like(w)) + g * g
                s["G"] = G
                w -= self.lr * g / (np.sqrt(G) + 1e-10)
            elif self.kind == "rmsprop":
                E = self.rho * s.get("E", np.zeros_like(w)) + (1 - self.rho) * g * g
                s["E"] = E
                w -= self.lr * g / (np.sqrt(E) + self.eps)
            elif self.kind in ("adam", "adamw"):
                m = self.b1 * s.get("m", np.zeros_like(w)) + (1 - self.b1) * g
                v = self.b2 * s.get("v", np.zeros_like(w)) + (1 - self.b2) * g * g
                s["m"], s["v"] = m, v
                m_hat, v_hat = m / (1 - self.b1 ** self.t), v / (1 - self.b2 ** self.t)
                if self.kind == "adamw" and decay:
                    w -= self.lr * decay * w                         # decoupled weight decay (Loshchilov & Hutter)
                w -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
            else:
                raise ValueError(self.kind)


def train_mlp(X_tr, y_tr, X_va, y_va, kind, lr, epochs, weight_decay=0.0, seed=0, hidden=128, batch=64):
    # Returns (model, history[epoch] = (train loss on a 1,500-row sample, val loss, val accuracy)).
    model = TinyMLP(X_tr.shape[1], hidden, int(y_tr.max()) + 1, seed)
    opt = Optimizer(kind, lr, weight_decay)
    rng = np.random.default_rng(seed + 1000)
    probe = rng.choice(len(X_tr), size=min(1500, len(X_tr)), replace=False)
    hist = []
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        for _ in range(epochs):
            order = rng.permutation(len(X_tr))
            for i in range(0, len(order), batch):
                idx = order[i:i + batch]
                opt.step(model.p, model.loss_and_grads(X_tr[idx], y_tr[idx])[1])
            P = model.predict_proba(X_va)
            va = float(-np.log(np.clip(P[np.arange(len(y_va)), y_va], 1e-12, None)).mean())
            trl = float(model.loss_and_grads(X_tr[probe], y_tr[probe])[0])
            hist.append((trl, va, float(np.mean(P.argmax(1) == y_va))))
            if not np.isfinite(va) or not np.isfinite(trl):
                break
    return model, np.array(hist)


# finite-difference gradient check on a tiny random problem
_rng = np.random.default_rng(0)
Xg, yg = _rng.normal(size=(7, 5)), _rng.integers(0, 3, 7)
net = TinyMLP(5, 4, 3, 0)
_, analytic = net.loss_and_grads(Xg, yg)
worst = 0.0
for k in net.p:
    for idx in np.ndindex(net.p[k].shape):
        orig = net.p[k][idx]
        net.p[k][idx] = orig + 1e-6
        lp = net.loss_and_grads(Xg, yg)[0]
        net.p[k][idx] = orig - 1e-6
        lm = net.loss_and_grads(Xg, yg)[0]
        net.p[k][idx] = orig
        worst = max(worst, abs((lp - lm) / 2e-6 - analytic[k][idx]))
print(f"gradient check: max |numeric - analytic| = {worst:.2e}")
assert worst < 1e-6

svd_opt = TruncatedSVD(n_components=N_SVD, random_state=SEED).fit(X_otr)
lsa_opt = StandardScaler().fit(svd_opt.transform(X_otr))
Z_otr, Z_ova = lsa_opt.transform(svd_opt.transform(X_otr)), lsa_opt.transform(svd_opt.transform(X_ova))
yi_otr = np.array([CLASSES.index(c) for c in y_otr])
yi_ova = np.array([CLASSES.index(c) for c in y_ova])
print(f"LSA-{N_SVD} keeps {svd_opt.explained_variance_ratio_.sum():.1%} of TF-IDF variance")

#### Learning-rate sensitivity

Each optimizer is swept over 5 learning rates around its usual range, for `OPT_EPOCHS` epochs, with no weight decay
(AdamW uses a light wd = 0.01, otherwise it would be identical to Adam).
The score is the **best validation log-loss over epochs** (an early-stopping oracle), averaged over seeds.

A **wide dark band** means the optimizer is forgiving about its learning rate, which matters because you rarely
get to tune much on device-sized budgets.

In [ ]:
SWEEP = {
    "sgd": [0.01, 0.03, 0.1, 0.3, 1.0],
    "momentum": [0.001, 0.003, 0.01, 0.03, 0.1],
    "nesterov": [0.001, 0.003, 0.01, 0.03, 0.1],
    "adagrad": [0.003, 0.01, 0.03, 0.1, 0.3],
    "rmsprop": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    "adam": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
    "adamw": [1e-4, 3e-4, 1e-3, 3e-3, 1e-2],
}
sweep_rows, t0 = [], time.time()
for kind, lrs in SWEEP.items():
    for lr in lrs:
        bests, best_eps = [], []
        for seed in OPT_SEEDS:
            _, hist = train_mlp(Z_otr, yi_otr, Z_ova, yi_ova, kind, lr, epochs=OPT_EPOCHS,
                                weight_decay=0.01 if kind == "adamw" else 0.0, seed=seed)
            va = hist[:, 1]
            finite = np.isfinite(va)
            bests.append(float(va[finite].min()) if finite.any() else np.inf)
            best_eps.append(int(np.argmin(np.where(finite, va, np.inf))) + 1 if finite.any() else 0)
        sweep_rows.append({"optimizer": kind, "lr": lr, "best val log-loss": float(np.mean(bests)),
                           "best epoch": float(np.mean(best_eps))})
sweep = pd.DataFrame(sweep_rows)
print(f"sweep: {len(sweep)} settings x {len(OPT_SEEDS)} seed(s) in {time.time() - t0:.0f}s")

kinds = list(SWEEP)
grid = np.array([[sweep[(sweep.optimizer == k) & (sweep.lr == lr)]["best val log-loss"].iloc[0] for lr in SWEEP[k]] for k in kinds])
shown = np.where(np.isfinite(grid), np.minimum(grid, 1.2), np.nan)
fig, ax = plt.subplots(figsize=(10, 4.8))
im = ax.imshow(shown, cmap=BLUE_GOOD, aspect="auto", vmin=np.nanmin(shown), vmax=np.nanmax(shown))
for i, k in enumerate(kinds):
    for j, lr in enumerate(SWEEP[k]):
        val = grid[i, j]
        dark = np.isfinite(val) and (min(val, 1.2) - np.nanmin(shown)) < 0.5 * (np.nanmax(shown) - np.nanmin(shown))
        ax.text(j, i, f"lr {lr:g}\n{val:.3f}" if np.isfinite(val) else f"lr {lr:g}\ndiverged", ha="center", va="center",
                fontsize=8, color="#ffffff" if dark else INK)
ax.set_yticks(range(len(kinds)), kinds)
ax.set_xticks(range(5), ["lowest lr", "", "middle", "", "highest lr"])
ax.grid(False)
cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label("best validation log-loss (darker = better; capped at 1.2)", color=INK_2)
cbar.outline.set_visible(False)
finish(ax, "Learning-rate sensitivity per optimizer", f"MLP on LSA-{N_SVD}, best epoch of {OPT_EPOCHS}, mean of {len(OPT_SEEDS)} seed(s)")
plt.tight_layout()
plt.show()

best_lr = sweep.loc[sweep.groupby("optimizer")["best val log-loss"].idxmin()].set_index("optimizer").reindex(kinds)
global_best = best_lr["best val log-loss"].min()
best_lr["lrs within 5% of global best"] = [int((sweep[(sweep.optimizer == k)]["best val log-loss"] <= 1.05 * global_best).sum()) for k in kinds]
best_lr.round(4)

#### Training dynamics at each optimizer's best learning rate

Mean over seeds. The **gap** between the train curve (grey) and the validation curve (violet) is overfitting. The
epoch where validation bottoms out is where early stopping should cut.

In [ ]:
curve_hist = {}
for kind in kinds:
    lr = float(best_lr.loc[kind, "lr"])
    hs = [train_mlp(Z_otr, yi_otr, Z_ova, yi_ova, kind, lr, epochs=OPT_EPOCHS,
                    weight_decay=0.01 if kind == "adamw" else 0.0, seed=s)[1] for s in CURVE_SEEDS]
    n = min(len(h) for h in hs)
    curve_hist[kind] = np.mean([h[:n] for h in hs], axis=0)

fig, axes = plt.subplots(2, 4, figsize=(14, 6.2), sharey=True)
for ax, kind in zip(axes.ravel(), kinds):
    h = curve_hist[kind]
    ep = np.arange(1, len(h) + 1)
    ax.plot(ep, h[:, 0], color=MUTED, label="train")
    ax.plot(ep, h[:, 1], color=STAT_C, label="validation")
    ax.scatter(np.argmin(h[:, 1]) + 1, h[:, 1].min(), s=40, color=STAT_C, edgecolor=SURFACE, linewidth=2, zorder=3)
    ax.set_ylim(0, 1.0)
    finish(ax, f"{kind} (lr {float(best_lr.loc[kind, 'lr']):g})")
axes[0, 0].legend(loc="upper right")
axes[1, 3].axis("off")
axes[1, 3].text(0, 0.6, "dot = best validation epoch\n\nevery optimizer overfits:\nearly stopping + weight decay\nmatter more than the choice\nof optimizer here",
                fontsize=9, color=INK_2, va="top")
for ax in axes[1, :3]:
    ax.set_xlabel("epoch")
plt.tight_layout()
plt.show()

#### Regularisation: Adam + L2 vs AdamW (decoupled weight decay)

* **Adam + L2** adds `λ·w` to the gradient. Adam then divides it by `√v`, so weights with large gradient history are
  regularised *less*, which is not what L2 is supposed to do.
* **AdamW** shrinks every weight by `lr·wd·w` directly, outside the adaptive scaling (Loshchilov & Hutter, 2019).

The two decay knobs live on different scales, so each is swept over its own natural range at lr = 1e-3.

In [ ]:
DECAY = {"adam": [0.0, 1e-3, 1e-2, 3e-2, 1e-1], "adamw": [0.0, 0.1, 1.0, 3.0, 10.0]}
wd_rows = []
for kind, values in DECAY.items():
    for wd in values:
        res = [train_mlp(Z_otr, yi_otr, Z_ova, yi_ova, kind, 1e-3, epochs=OPT_EPOCHS, weight_decay=wd, seed=s)[1] for s in OPT_SEEDS]
        wd_rows.append({"optimizer": "Adam + L2 (coupled)" if kind == "adam" else "AdamW (decoupled)", "decay": wd,
                        "best val log-loss": float(np.mean([np.nanmin(h[:, 1]) for h in res])),
                        "final val log-loss": float(np.mean([h[-1, 1] for h in res])),
                        "final train log-loss": float(np.mean([h[-1, 0] for h in res]))})
wd_table = pd.DataFrame(wd_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.9), sharey=True)
for ax, name in zip(axes, ["Adam + L2 (coupled)", "AdamW (decoupled)"]):
    part = wd_table[wd_table.optimizer == name]
    x = np.arange(len(part))
    ax.plot(x, part["final val log-loss"], color=MUTED, marker="o", label=f"final (epoch {OPT_EPOCHS})")
    ax.plot(x, part["best val log-loss"], color=STAT_C, marker="o", label="best epoch")
    ax.set_xticks(x, [f"{v:g}" for v in part.decay])
    ax.set_xlabel("λ (L2 coefficient)" if "L2" in name else "weight decay (× lr per step)")
    finish(ax, name)
axes[0].set_ylabel("validation log-loss")
axes[0].legend(loc="upper right")
plt.tight_layout()
plt.show()
best_l2 = wd_table[wd_table.optimizer.str.startswith("Adam +")].nsmallest(1, "best val log-loss").iloc[0]
best_w = wd_table[wd_table.optimizer.str.startswith("AdamW")].nsmallest(1, "best val log-loss").iloc[0]
print(f"best Adam+L2: λ={best_l2.decay:g} -> {best_l2['best val log-loss']:.4f} | "
      f"best AdamW: wd={best_w.decay:g} -> {best_w['best val log-loss']:.4f}")
wd_table.round(4)

In [ ]:
# Full-batch quasi-Newton on a non-convex net: scikit-learn's L-BFGS MLP (no minibatches, no early stopping).
t = time.time()
lbfgs_mlp = MLPClassifier(hidden_layer_sizes=(128,), solver="lbfgs", alpha=1e-4, max_iter=200, random_state=SEED).fit(Z_otr, y_otr)
print(f"L-BFGS MLP: {time.time() - t:.1f}s, {lbfgs_mlp.n_iter_} iterations | train log-loss "
      f"{log_loss(y_otr, lbfgs_mlp.predict_proba(Z_otr), labels=CLASSES):.4f} | validation log-loss "
      f"{log_loss(y_ova, lbfgs_mlp.predict_proba(Z_ova), labels=CLASSES):.4f}")
print(f"reference - best minibatch optimizer above: {global_best:.4f}; sparse TF-IDF LR (L-BFGS): {ref_loss:.4f}")

#### Optimizer recommendations

| Model | Optimizer | Why |
|---|---|---|
| **TF-IDF logistic regression** (the likely detector) | **L-BFGS** (or newton-cg) | convex: every solver reaches the same model, so pick the fast deterministic one; no learning rate to tune |
| L1 / elastic-net LR, to shrink the on-device model | **SAGA** | the only scikit-learn solver with L1 for multinomial; expect it to be slower |
| On-device incremental updates from user feedback | **SGD, constant step + averaging**, warm-started | cheap per message; averaging makes unattended updates stable |
| Small MLP / CNN on dense features | **AdamW** (or Adam + tuned L2) with early stopping; SGD + Nesterov momentum if you can afford LR tuning | adaptive methods tolerate a wider LR range; with tuning, momentum SGD generalises as well; overfitting dominates, so decay + early stopping matter more than the optimizer |
| **Fine-tuning a small transformer** (DistilBERT / MiniLM / MobileBERT) | **AdamW**, lr 2e-5 to 5e-5, wd 0.01, linear warm-up 6–10 % then linear decay, grad-clip 1.0 | transformer gradients vary by orders of magnitude across layers (Adam's per-parameter scaling handles that); decoupled decay keeps the regularisation strength independent of the LR schedule; warm-up protects pre-trained weights while Adam's moment estimates are still poor |
| Avoid | AdaGrad (step sizes decay to nothing); L-BFGS for neural nets (full-batch, overfits, no early stopping); plain SGD without momentum (slow, very LR-sensitive) | see the sweep and the L-BFGS MLP result above |

## 10 · Held-out test: the selected model (and the leader, if different)

Each model is refit on all of `dev` and scored **once** on the grouped test split. Calibrated families get a grouped
inner split for their calibrator.

In [ ]:
final_blocks = fold_features(dev, test)
inner_cv_dev = list(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(np.zeros(len(dev)), y_dev, g_dev))
w_test = 1.0 / pd.Series(g_test).map(pd.Series(g_test).value_counts()).to_numpy()
final_models, test_rows = {}, []
for family in dict.fromkeys([CHOSEN, LEADER]):
    params = json.loads(best_config[family])
    X_d, X_t = final_blocks[FAMILY_BLOCK[family]]
    model = make_estimator(family, params, inner_cv_dev).fit(X_d, y_dev)
    final_models[family] = (model, X_t)
    test_rows.append({"family": family, "config": best_config[family], **evaluate(model, X_t, y_test, w_test)})
test_table = pd.DataFrame(test_rows).set_index("family")
display(test_table.style.format(precision=4, na_rep="n/a"))

model, X_t = final_models[CHOSEN]
yhat_test = model.predict(X_t)
boot = []
for _ in range(1000):
    i = RNG.integers(0, len(y_test), len(y_test))
    boot.append(f1_score(y_test[i], yhat_test[i], labels=CLASSES, average="macro", zero_division=0))
print(f"{CHOSEN}: test macro-F1 {f1_score(y_test, yhat_test, labels=CLASSES, average='macro'):.4f} "
      f"(bootstrap 95% CI {np.percentile(boot, 2.5):.4f} - {np.percentile(boot, 97.5):.4f})")

cm = confusion_matrix(y_test, yhat_test, labels=CLASSES)
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.imshow(cm, cmap=BLUE_SEQ)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f"{cm[i, j]}", ha="center", va="center", color="#ffffff" if cm[i, j] > cm.max() / 2 else INK)
ax.set_xticks(range(3), CLASSES)
ax.set_yticks(range(3), CLASSES)
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.grid(False)
finish(ax, "Confusion matrix, test", CHOSEN)
plt.tight_layout()
plt.show()

### 10.1 · Calibration (top-label reliability)

In [ ]:
P_test = model.predict_proba(X_t)
conf = P_test.max(axis=1)
correct = (np.array(model.classes_, dtype=object)[P_test.argmax(axis=1)] == y_test).astype(float)
bins = np.clip(((conf - 1 / 3) / (2 / 3) * 10).astype(int), 0, 9)    # confidence lives in [1/3, 1]
rel = pd.DataFrame({"conf": conf, "correct": correct, "bin": bins}).groupby("bin").agg(
    mean_conf=("conf", "mean"), accuracy=("correct", "mean"), n=("correct", "size"))
ece = float(np.sum(rel.n * (rel.mean_conf - rel.accuracy).abs()) / len(conf))
fig, ax = plt.subplots(figsize=(5.6, 4.8))
ax.plot([1 / 3, 1], [1 / 3, 1], color=AXIS, lw=1)
dense = rel[rel.n >= 20]
sparse_bins = rel[rel.n < 20]
ax.scatter(dense.mean_conf, dense.accuracy, s=30 + 3 * np.sqrt(dense.n), color=STAT_C, edgecolor=SURFACE, linewidth=2, label="bin with n >= 20")
ax.scatter(sparse_bins.mean_conf, sparse_bins.accuracy, s=36, facecolor=SURFACE, edgecolor=STAT_C, linewidth=1.5, label="bin with n < 20")
for _, r in rel.iterrows():
    ax.annotate(f"{int(r.n)}", (r.mean_conf, r.accuracy), xytext=(5, -11), textcoords="offset points", color=MUTED, fontsize=7.5)
ax.set_xlabel("mean top-label confidence (label = messages)")
ax.set_ylabel("accuracy")
ax.legend(loc="upper left")
finish(ax, "Reliability, test", f"{CHOSEN} · ECE {ece:.3f} · log-loss {log_loss(y_test, P_test, labels=list(model.classes_)):.4f}")
plt.show()

### 10.2 · Confidence gating: when may the assistant explain?

The project plan's first risk is that **a wrong verdict produces a confident wrong explanation**. The risk–coverage
curve shows how accuracy rises when the assistant only explains messages above a confidence threshold. Messages
below the threshold get a neutral *"can't tell, verify independently"* response.

In [ ]:
order = np.argsort(-conf)
acc_at = np.cumsum(correct[order]) / np.arange(1, len(order) + 1)
coverage = np.arange(1, len(order) + 1) / len(order)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(coverage, acc_at, color=STAT_C)
ax.set_xlabel("coverage (share of messages the assistant explains)")
ax.set_ylabel("accuracy on covered messages")
ax.set_ylim(max(0.5, acc_at.min() - 0.02), 1.005)
finish(ax, "Risk-coverage: accuracy vs share of messages explained", f"test set, {CHOSEN}")
plt.show()
gate_rows = []
for target in (0.95, 0.97, 0.99):
    ok = np.flatnonzero(acc_at >= target)
    k = int(ok.max()) if len(ok) else -1
    gate_rows.append({"target accuracy": target, "coverage": coverage[k] if k >= 0 else 0.0,
                      "confidence threshold": float(conf[order][k]) if k >= 0 else np.nan})
pd.DataFrame(gate_rows).round(3)

### 10.3 · Evidence faithfulness: deletion test

If the tokens shown to the user as evidence really drive the verdict, deleting them should hurt the predicted
class's probability **far more than deleting random tokens**. The test runs on the TF-IDF LR, because it is the
model whose token contributions would be displayed. Attribution of a word = its centred multinomial coefficient for
the predicted class.

In [ ]:
word_vec, char_vec = make_word_tfidf().fit(dev.model_text), make_char_tfidf().fit(dev.model_text)
featurize = lambda texts: sparse.hstack([word_vec.transform(texts), char_vec.transform(texts)]).tocsr()
explainer = LogisticRegression(C=C_BEST, max_iter=3000).fit(featurize(dev.model_text), y_dev)
n_word = len(word_vec.vocabulary_)
coef_centered = explainer.coef_[:, :n_word] - explainer.coef_[:, :n_word].mean(axis=0, keepdims=True)
vocab = word_vec.vocabulary_
exp_classes = list(explainer.classes_)

sample = test[test.label != "ham"].sample(min(300, int((test.label != "ham").sum())), random_state=SEED)
del_rows = []
for text in sample.model_text:
    tokens = re.findall(r"\b\w+\b", text)
    uniq = list(dict.fromkeys(tokens))
    if len(uniq) < 6:
        continue
    base_text = " ".join(tokens)
    p0 = explainer.predict_proba(featurize([base_text]))[0]
    c = int(p0.argmax())
    score = {t: coef_centered[c, vocab[t]] if t in vocab else 0.0 for t in uniq}
    ranked = sorted(uniq, key=lambda t: -score[t])
    for k in (1, 3, 5):
        top_drop = p0[c] - explainer.predict_proba(featurize([" ".join(t for t in tokens if t not in set(ranked[:k]))]))[0][c]
        rand_drops = []
        for r in range(3):
            removed = set(np.random.default_rng(r).choice(uniq, size=k, replace=False))
            rand_drops.append(p0[c] - explainer.predict_proba(featurize([" ".join(t for t in tokens if t not in removed)]))[0][c])
        del_rows.append({"k": k, "drop (top-attributed tokens)": top_drop, "drop (random tokens)": float(np.mean(rand_drops)),
                         "class flipped (top)": float(
                             explainer.predict_proba(featurize([" ".join(t for t in tokens if t not in set(ranked[:k]))]))[0].argmax() != c)})
deletion = pd.DataFrame(del_rows).groupby("k").mean()
print(f"deletion test on {len(del_rows) // 3} malicious test messages (explainer classes: {exp_classes})")
deletion.round(3)

## 11 · Next model to try: a small fine-tuned transformer (optional)

Try this **only after** the linear model plateaus on a fresh, non-augmented test set. Fine-tuning on a corpus where
a third of the malicious rows are synonym-swapped paraphrases mostly teaches the model those paraphrases.

| Choice | Value | Why |
|---|---|---|
| Model | `distilbert-base-uncased` (66 M) or a MiniLM-L6 variant (≈ 22 M); a multilingual/Indic encoder if code-mixed text matters | small enough for INT8 on a Pi 5 or phone; *uncased* matches the case-artifact fix |
| Input | `model_text`, max 128 tokens | entity tokens stop it memorising phone numbers; 128 covers multi-part SMS |
| Loss | cross-entropy, no class weights | calibrated probabilities; set thresholds afterwards |
| Optimizer | **AdamW**, lr 3e-5, wd 0.01 (not on bias/LayerNorm), β = (0.9, 0.999) | see §9 |
| Schedule | linear warm-up for 10 % of steps, then linear decay to 0 | protects pre-trained weights early; anneals at the end |
| Batch / epochs | 32 / up to 4, **early stopping on grouped-val macro-F1** | small data overfits within a few epochs (§9.3) |
| Clip | global gradient norm 1.0 | guards against early loss spikes |
| Export | `torch.export` → ExecuTorch (XNNPACK, INT8 PT2E quantisation) | same runtime as the Llama explainer |

The cell below is **not executed by default** (`RUN_TRANSFORMER = False`) and needs `torch` and `transformers`
installed.

In [ ]:
if RUN_TRANSFORMER:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

    MODEL_NAME, MAX_LEN, BATCH, EPOCHS, LR, WD, WARMUP = "distilbert-base-uncased", 128, 32, 4, 3e-5, 0.01, 0.1
    torch.manual_seed(SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    net = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(CLASSES)).to(device)

    tr_i, va_i = next(StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED).split(dev, y_dev, g_dev))
    def batches(texts, labels, shuffle):
        idx = np.random.default_rng(SEED).permutation(len(texts)) if shuffle else np.arange(len(texts))
        for i in range(0, len(idx), BATCH):
            j = idx[i:i + BATCH]
            enc = tok([texts[k] for k in j], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
            yield {k: v.to(device) for k, v in enc.items()}, torch.tensor([CLASSES.index(labels[k]) for k in j], device=device)

    tr_texts, tr_labels = dev.model_text.iloc[tr_i].tolist(), y_dev[tr_i]
    va_texts, va_labels = dev.model_text.iloc[va_i].tolist(), y_dev[va_i]
    no_decay = ("bias", "LayerNorm.weight")
    groups_ = [{"params": [p for n, p in net.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": WD},
               {"params": [p for n, p in net.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0}]
    optimizer = torch.optim.AdamW(groups_, lr=LR, betas=(0.9, 0.999), eps=1e-8)
    steps = EPOCHS * math.ceil(len(tr_texts) / BATCH)
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP * steps), steps)

    def predict(texts):
        net.eval()
        out = []
        with torch.no_grad():
            for enc, _ in batches(texts, np.array(["ham"] * len(texts), dtype=object), shuffle=False):
                out.append(torch.softmax(net(**enc).logits, dim=-1).cpu().numpy())
        return np.vstack(out)

    best_f1, best_state = -1.0, None
    for epoch in range(EPOCHS):
        net.train()
        for enc, labels in batches(tr_texts, tr_labels, shuffle=True):
            loss = torch.nn.functional.cross_entropy(net(**enc).logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        P_va = predict(va_texts)
        f1 = f1_score(va_labels, np.array(CLASSES, dtype=object)[P_va.argmax(1)], average="macro")
        print(f"epoch {epoch + 1}: val macro-F1 {f1:.4f}, val log-loss {log_loss(va_labels, P_va, labels=CLASSES):.4f}")
        if f1 > best_f1:
            best_f1, best_state = f1, {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
    net.load_state_dict(best_state)
    P_te = predict(test.model_text.tolist())
    print(f"TEST macro-F1 {f1_score(y_test, np.array(CLASSES, dtype=object)[P_te.argmax(1)], average='macro'):.4f}, "
          f"log-loss {log_loss(y_test, P_te, labels=CLASSES):.4f}")
else:
    print("RUN_TRANSFORMER is False - skipped (set it to True with torch + transformers installed).")

## 12 · Recommendations and export

In [ ]:
fam_order = list(summary.index)
lines = [
    f"**Data**: {len(raw):,} rows → {len(df):,} unique texts; {len(conflict_rows)} texts dropped for carrying both "
    f"spam and smishing labels; the advertised balance comes from copies and synonym-swap augmentation.",
    "**Model ranking (CV macro-F1)**: " + " > ".join(f"{f} {summary.loc[f, ('macro-F1', 'mean')]:.3f}" for f in fam_order[:5]) + ".",
    f"**Selected**: {CHOSEN} {CHOSEN_PARAMS}; tied with the leader ({LEADER}) and best calibrated among the tied "
    f"probabilistic families (CV log-loss {summary.loc[CHOSEN, ('log-loss', 'mean')]:.3f}).",
    f"**Test**: macro-F1 {test_table.loc[CHOSEN, 'macro-F1']:.3f}, group-weighted {test_table.loc[CHOSEN, 'group-weighted macro-F1']:.3f}, "
    f"malicious recall at ≤1% ham FPR {test_table.loc[CHOSEN, 'recall@ham-FPR≤1%']:.3f}, ECE {ece:.3f}.",
    f"**Optimizers**: LR solvers agree to {solver_table['max |Δp| vs lbfgs'].drop('liblinear (one-vs-rest)').max():.1e}; "
    f"fastest {solver_table['fit s'].idxmin()} ({solver_table['fit s'].min():.1f}s). MLP: best {best_lr['best val log-loss'].idxmin()} "
    f"({global_best:.3f}); sparse LR reaches {ref_loss:.3f}, so the linear model wins here.",
]
display(Markdown("\n".join(f"{i}. {line}" for i, line in enumerate(lines, 1))))

out_cols = ["message", "label", "n_copies", "group", "split", "model_text", "URL", "EMAIL", "PHONE"] + sp.NUM_COLS
(PROJECT_ROOT / "data" / "processed").mkdir(parents=True, exist_ok=True)
df[out_cols].to_csv(PROJECT_ROOT / "data" / "processed" / "balanced_smishing_clean_splits.csv", index=False, encoding="utf-8")
results = {
    "data_sha256": DATA_SHA256, "environment": ENV, "quick_mode": QUICK,
    "cv_summary_mean": json.loads(summary.xs("mean", axis=1, level=1).to_json(orient="index")),
    "selection": {"leader": LEADER, "tied": tied, "selected": CHOSEN, "params": CHOSEN_PARAMS},
    "test": json.loads(test_table.to_json(orient="index")),
    "lr_solvers": json.loads(solver_table.to_json(orient="index")),
    "sgd_schedules_final_val_logloss": {f"{s}{' avg' if a else ''}": v[-1] for (s, a), v in sgd_curves.items()},
    "mlp_optimizer_sweep": json.loads(sweep.to_json(orient="records")),
    "weight_decay_study": json.loads(wd_table.to_json(orient="records")),
}
(OUT_DIR / "results.json").write_text(json.dumps(results, indent=1, default=str), encoding="utf-8")
print("wrote data/processed/balanced_smishing_clean_splits.csv and", (OUT_DIR / "results.json").relative_to(PROJECT_ROOT))

### What to try, in order

1. **TF-IDF (word 1–2 + char_wb 2–5) + logistic regression, L-BFGS.** Expect it to be tied for best and best
   calibrated. Ship it as the detector. It also provides the token evidence (§10.3).
2. **Calibrated Complement NB / linear SVM.** Use these if either leads on macro-F1. Calibration makes their
   confidences usable for gating (§10.2).
3. **A two-stage design.** First a high-precision ham-vs-malicious detector, then a spam-vs-smishing subtype model
   (§8 framing table). The noisy subtype boundary then never costs a warning.
4. **SGD (log-loss, constant step + averaging)**, warm-started from the LR weights, for **on-device feedback updates**.
5. **Small transformer + AdamW** (§11). Only once you have a fresh, non-augmented, Indian-context test set where the
   linear model plateaus.
6. **Skip** MLP-on-LSA and gradient boosting for this text task. LSA throws away about half the TF-IDF signal, and
   trees on dense text projections trail the sparse linear models. Keep boosting for tabular problems (URL lexical
   features, network flows).

**Data actions before trusting any metric:**
* evaluate on non-augmented messages;
* re-label or adjudicate the spam/smishing conflicts (§7.4);
* add Indian transactional hard negatives (OTP, debit alerts);
* define the scam **subtype** taxonomy (KYC, delivery fee, prize, job, loan, …) as new labels. This dataset only
  separates spam from smishing.